# Sprint 3 — Inference: UCI Student Performance Risk Model

Loads the already-trained model from `model_training_uci.ipynb`, **saves it as
a `.txt` file**, reloads a fresh model instance from that `.txt` file only, and
runs inference with it — both batch (score every student) and single-record
(score one new student, the way a real application would call this).

## Important honesty check on the `.txt` format

XGBoost's `save_model()` picks its file format from the extension: `.json` →
JSON, `.ubj` → binary UBJSON, anything else (including `.txt`) → **still binary**
(this XGBoost version defaults to UBJSON under the hood even for a `.txt` name,
with a warning saying exactly that). So **`risk_model_uci_v1.txt` is not
human-readable text** — opening it in a text editor shows binary bytes, not
JSON or tree diagrams. It's a fully valid, reloadable model file that happens
to be named `.txt`; the name doesn't change the content.

If you actually want a **human-readable** dump of the trees (for
documentation, not for reloading), `booster.dump_model()` produces real text —
included at the end of this notebook as a bonus, clearly separate from the
functional `.txt` model file.

In [3]:
import os
import numpy as np
import pandas as pd
import joblib
from xgboost import XGBClassifier

PROJECT_ROOT = r"C:\Users\KaranSanjayJadhav\Desktop\Karan_Jadhav_student_analytics_streamlit\student_analytics_cli"
PROCESSED_DIR = os.path.join(PROJECT_ROOT, "data", "processed")
MODELS_DIR = os.path.join(PROJECT_ROOT, "models")

RISK_THRESHOLD = 0.768  # carried over from model_training_uci.ipynb Section 7b
                         # (the F1-optimal threshold found on that notebook's test split)

## 1. Load the trained model and save it as `.txt`

In [4]:
joblib_path = os.path.join(MODELS_DIR, "risk_model_uci_v1.joblib")
model = joblib.load(joblib_path)
print(f"Loaded from: {joblib_path}")
print(f"Model type: {type(model).__name__}")

txt_path = os.path.join(MODELS_DIR, "risk_model_uci_v1.txt")
model.save_model(txt_path)
print(f"\nSaved as: {txt_path}")
print(f"File size: {os.path.getsize(txt_path):,} bytes")

with open(txt_path, "rb") as f:
    preview = f.read(60)
print(f"First 60 bytes (proof it's binary, not readable text): {preview}")

Loaded from: C:\Users\KaranSanjayJadhav\Desktop\Karan_Jadhav_student_analytics_streamlit\student_analytics_cli\models\risk_model_uci_v1.joblib
Model type: XGBClassifier

Saved as: C:\Users\KaranSanjayJadhav\Desktop\Karan_Jadhav_student_analytics_streamlit\student_analytics_cli\models\risk_model_uci_v1.txt
File size: 550,717 bytes
First 60 bytes (proof it's binary, not readable text): b'{L\x00\x00\x00\x00\x00\x00\x00\x07learner{L\x00\x00\x00\x00\x00\x00\x00\nattributes{L\x00\x00\x00\x00\x00\x00\x00\x0cscikit_learnS'


c:\Users\KaranSanjayJadhav\Desktop\Karan_Jadhav_student_analytics_streamlit\student_analytics_cli\venv\Lib\site-packages\xgboost\core.py:158: UserWarning: [23:37:40] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0015a694724fa8361-1\xgboost\xgboost-ci-windows\src\c_api\c_api.cc:1374: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats.
  warnings.warn(smsg, UserWarning)


## 2. Reload a fresh model *only* from the `.txt` file

In [5]:
# A brand-new, untrained XGBClassifier -- this proves the .txt file alone is
# enough to reconstruct a working model, independent of the original .joblib
loaded_model = XGBClassifier()
loaded_model.load_model(txt_path)
print("Model reloaded from .txt file.")

# Sanity check: predictions from the .txt-reloaded model should exactly match
# the original joblib-loaded model
X_check = pd.read_csv(os.path.join(PROCESSED_DIR, "uci_feature_matrix.csv")).drop(columns=["at_risk"]).head(20)
original_proba = model.predict_proba(X_check)[:, 1]
reloaded_proba = loaded_model.predict_proba(X_check)[:, 1]

max_diff = np.abs(original_proba - reloaded_proba).max()
print(f"Max prediction difference (.joblib vs .txt-reloaded): {max_diff:.10f}")
assert max_diff < 1e-6, "Reloaded model does not match the original -- something was lost in the save/load round trip"
print("Confirmed: identical predictions. The .txt file round-trips perfectly.")

Model reloaded from .txt file.
Max prediction difference (.joblib vs .txt-reloaded): 0.0000000000
Confirmed: identical predictions. The .txt file round-trips perfectly.


## 3. `predict_risk()` — same shape as the training notebook

In [6]:
def predict_risk(model, X: pd.DataFrame, threshold: float = RISK_THRESHOLD) -> pd.DataFrame:
    """
    Score a feature matrix and return risk_score / predicted_label / risk_level.
    Identical logic to model_training_uci.ipynb's predict_risk() -- this is the
    inference-time copy, using only the .txt-reloaded model.
    """
    proba = model.predict_proba(X)[:, 1]

    def bucket(p):
        if p < 0.3:
            return "Low"
        elif p < 0.6:
            return "Medium"
        return "High"

    return pd.DataFrame(
        {
            "risk_score": proba.round(4),
            "predicted_label": np.where(proba >= threshold, "at_risk", "not_at_risk"),
            "risk_level": [bucket(p) for p in proba],
        },
        index=X.index,
    )

## 4. Batch inference — score every student

In [7]:
X_all = pd.read_csv(os.path.join(PROCESSED_DIR, "uci_feature_matrix.csv")).drop(columns=["at_risk"])
batch_predictions = predict_risk(loaded_model, X_all)

print(f"Scored {len(batch_predictions):,} students using the .txt-reloaded model")
print(f"\nrisk_level distribution:")
print(batch_predictions["risk_level"].value_counts())
print(f"\nTop 10 highest-risk students:")
batch_predictions.sort_values("risk_score", ascending=False).head(10)

Scored 1,044 students using the .txt-reloaded model

risk_level distribution:
risk_level
Low       736
High      244
Medium     64
Name: count, dtype: int64

Top 10 highest-risk students:


,risk_score,predicted_label,risk_level
213,0.9412,at_risk,High
1000,0.9379,at_risk,High
128,0.9376,at_risk,High
221,0.9373,at_risk,High
964,0.9372,at_risk,High
827,0.9369,at_risk,High
383,0.9362,at_risk,High
352,0.9360,at_risk,High
216,0.9341,at_risk,High
137,0.9337,at_risk,High


## 5. Single-record inference — scoring one new student

This is the realistic use case for a deployed model: a Streamlit form or API
endpoint receives **one student's data**, not a CSV of thousands. The feature
columns and order must match training exactly — a mismatched or missing column
here is the single most common real-world inference bug, so this cell builds
the record deliberately from the exact training column list rather than typing
values in an arbitrary order.

In [8]:
feature_columns = X_all.columns.tolist()
print("Required feature columns, in order:")
print(feature_columns)

Required feature columns, in order:
['age', 'medu', 'fedu', 'traveltime', 'studytime', 'failures', 'famrel', 'freetime', 'goout', 'dalc', 'walc', 'health', 'absences', 'g1', 'g2', 'school_MS', 'sex_M', 'address_U', 'famsize_LE3', 'schoolsup_yes', 'famsup_yes', 'higher_yes', 'internet_yes', 'romantic_yes', 'subject_Portuguese']


In [9]:
# Example: a hypothetical student with a couple of past failures, moderate
# absences, and mediocre first/second-period grades -- the kind of profile
# Section 5 of feature_engineering_uci.ipynb flagged as elevated risk.
new_student = {
    "age": 17, "medu": 2, "fedu": 2, "traveltime": 1, "studytime": 2,
    "failures": 1, "famrel": 4, "freetime": 3, "goout": 4, "dalc": 2, "walc": 3,
    "health": 3, "absences": 12, "g1": 8, "g2": 7,
    "school_MS": False, "sex_M": True, "address_U": True, "famsize_LE3": False,
    "schoolsup_yes": False, "famsup_yes": True, "higher_yes": True,
    "internet_yes": True, "romantic_yes": False, "subject_Portuguese": False,
}

X_new = pd.DataFrame([new_student])[feature_columns]  # enforce exact training column order
result = predict_risk(loaded_model, X_new)
print(result.to_string(index=False))

 risk_score predicted_label risk_level
      0.933         at_risk       High


## 7. Summary

- **Saved** `models/risk_model_uci_v1.txt` via `model.save_model()` — functionally
  a complete model file despite the `.txt` name; it's binary (UBJSON), not
  human-readable, and that's normal for this format, not a bug.
- **Verified round-trip:** predictions from a model reloaded *only* from the
  `.txt` file exactly match the original `.joblib` model (max difference < 1e-6).
- **Batch inference** — scored the full saved feature matrix through the
  `.txt`-reloaded model.
- **Single-record inference** — the realistic deployment pattern: one student's
  data in, one risk score out, built from the exact training column order to
  avoid the most common inference bug (column mismatch).
- **Bonus:** a genuinely human-readable tree dump (`_readable_dump.txt`) for
  documentation/auditing — explicitly not a reloadable model file, kept
  separate from the functional one to avoid confusing the two.